# Supply Chain Logistics Optimization
**Multi-algorithm optimization: LP · Network Flow · Dijkstra · WSM · MILP**

Dataset: 9,215 shipment orders | Target: minimize total cost & late deliveries

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from supply_chain_optimizer import (
    SupplyChainDataLoader, LinearProgrammingOptimizer,
    NetworkFlowOptimizer, DijkstraRouter,
    WeightedScoringModel, PlantActivationMILP,
    ScenarioAnalyzer, run_optimization_pipeline
)

DATA_PATH = '../data/supply_chain_logistics_data.xlsx'
print('Imports OK')

## 1. Load & Explore Data

In [ ]:
loader = SupplyChainDataLoader(DATA_PATH)
df = loader.load()
summary = loader.summary()

print(f'Orders      : {summary["total_orders"]:,}')
print(f'Late orders : {summary["late_orders"]} ({100-summary["on_time_rate"]:.2f}%)')
print(f'On-time     : {summary["on_time_rate"]}%')
print(f'Baseline $  : ${summary["total_cost"]:,.0f}')
df.head()

In [ ]:
# Plant concentration
print('Plant order share:')
print(df['Plant Code'].value_counts(normalize=True).mul(100).round(1).to_string())

## 2. Linear Programming — Cost Minimization

In [ ]:
lp = LinearProgrammingOptimizer(df)
lp_result = lp.solve()

print(f'Status         : {lp_result["status"]}')
print(f'Baseline cost  : ${lp_result["baseline_cost"]:,.0f}')
print(f'Optimized cost : ${lp_result["optimized_cost"]:,.0f}')
print(f'Savings        : ${lp_result["savings"]:,.0f}  ({lp_result["savings_pct"]}%)')

## 3. Network Flow Model

In [ ]:
nf = NetworkFlowOptimizer(df)
nf_result = nf.solve()
print('Network summary:')
for k, v in nf_result.items():
    print(f'  {k}: {v}')

## 4. Dijkstra Route Prediction

In [ ]:
router = DijkstraRouter(df)
routes = router.best_routes(top_n=5)

print('Top 5 optimal routes:')
for i, r in enumerate(routes, 1):
    print(f'  {i}. {r["path"]}  |  Cost: ${r["cost"]:,.0f}')

## 5. Weighted Scoring Model — Carrier Selection

In [ ]:
wsm = WeightedScoringModel(df)
scores = wsm.score_carriers()
print(wsm.recommend())
scores

## 6. MILP — Plant Activation

In [ ]:
milp = PlantActivationMILP(df)
milp_result = milp.solve()

print('Activate  :', milp_result['plants_to_activate'])
print('Deactivate:', milp_result['plants_to_deactivate'])
pd.DataFrame(milp_result['plant_decisions'])

## 7. Scenario Analysis

In [ ]:
analyzer = ScenarioAnalyzer(df['Total_Cost'].sum())
scenarios = analyzer.run_all()
scenarios.sort_values('Cost_Change_pct')

## 8. Full Pipeline (One-Shot)

In [ ]:
results = run_optimization_pipeline(DATA_PATH)
print('Pipeline complete.')
print('LP savings:', results['lp']['savings_pct'], '%')